# All of Us Rare Variant Burden Analysis Pipeline

Complete workflow for gene-based rare variant burden testing using REGENIE.

**Pipeline Steps:**
1. Load All of Us dataset
2. Define cases and controls
3. Prepare phenotype and covariate files
4. Extract and filter genotypes by MAF
5. Create gene annotations
6. Run REGENIE Step 1 (null model)
7. Run REGENIE Step 2 (burden tests)

**Author:** Marzieh Khani

**Date:** 03/09/2026


## Create covar and pheno files

In [ ]:
import pandas as pd

# Load the original CSV file
file_path = 'AD_R8_dataset_person_df.csv'
df = pd.read_csv(file_path)

# Select the desired columns
selected_columns = ['person_id', 'gender', 'date_of_birth']
new_df = df[selected_columns]

# Map gender values
def map_gender(gender):
    if gender == 'Male':
        return 1
    elif gender == 'Female':
        return 2
    else:
        return -9

new_df['gender'] = new_df['gender'].apply(map_gender)

# Calculate age from date_of_birth
new_df['date_of_birth'] = pd.to_datetime(new_df['date_of_birth'])
current_date = pd.to_datetime('today')
new_df['age'] = current_date.year - new_df['date_of_birth'].dt.year

# Rename the columns
new_df.rename(columns={
    'person_id': 'FID',
    'gender': 'SEX'
}, inplace=True)

# Save the new DataFrame to a new CSV file
new_file_path = 'covar_AD_R8_file.csv'
new_df.to_csv(new_file_path, index=False)


In [ ]:
import pandas as pd

# Load the data
df = pd.read_csv('covar_AD_R8_file.csv')

# Remove duplicate rows based on the 'FID' column
df = df.drop_duplicates(subset=['FID'])

# Save the updated DataFrame
df.to_csv('covar_AD_R8_uniqID_file.csv', index=False)

In [ ]:
import pandas as pd

# Load the original CSV file
file_path = 'AD_R8_dataset_condition_df.csv'
df = pd.read_csv(file_path)

# Select the desired columns
selected_columns = ['person_id', 'condition_start_datetime']
new_df = df[selected_columns]

# Rename the columns
new_df.rename(columns={
    'person_id': 'FID',
    'condition_start_datetime': 'Age_at_Onset'
}, inplace=True)

# Save the new DataFrame to a new CSV file
new_file_path = 'filtered_condition_AD_R8_df.csv'
new_df.to_csv(new_file_path, index=False)


In [ ]:
import pandas as pd

# Load the data
df = pd.read_csv('filtered_condition_AD_R8_df.csv')

# Remove duplicate rows based on the 'FID' column
df = df.drop_duplicates(subset=['FID'])

# Save the updated DataFrame
df.to_csv('filtered_condition_AD_R8_uniqID_file.csv', index=False)

In [ ]:
import pandas as pd

# Load the first CSV file
file_path1 = 'covar_AD_R8_file.csv'
df1 = pd.read_csv(file_path1)

# Load the second CSV file
file_path2 = 'filtered_condition_AD_R8_uniqID_file.csv'
df2 = pd.read_csv(file_path2)

# Merge the DataFrames on the 'FID' column
merged_df = pd.merge(df1, df2, on='FID', how='inner')

# Save the merged DataFrame to a new CSV file
new_file_path = 'merged_covar_condition_AD_R8.csv'
merged_df.to_csv(new_file_path, index=False)


In [ ]:
import pandas as pd

# Load the merged CSV file
file_path = 'merged_covar_condition_AD_R8.csv'
df = pd.read_csv(file_path)

# Convert date_of_birth to datetime and extract the year
df['date_of_birth'] = pd.to_datetime(df['date_of_birth'], errors='coerce').dt.year

# Function to handle different date formats in Age_at_Onset
def parse_age_at_onset(date):
    try:
        return pd.to_datetime(date, errors='coerce').year
    except Exception as e:
        return None

# Apply the function to the Age_at_Onset column
df['Age_at_Onset'] = df['Age_at_Onset'].apply(parse_age_at_onset)

# Calculate the age at onset by subtracting the years
df['age_at_onset2'] = df['Age_at_Onset'] - df['date_of_birth']

# Remove the decimal part by converting to integer
df['Age_at_Onset'] = df['Age_at_Onset'].astype('Int64')
df['age_at_onset2'] = df['age_at_onset2'].astype('Int64')

# Save the updated DataFrame to a new CSV file
new_file_path = 'updated_merged_covar_condition_AD_R8.csv'
df.to_csv(new_file_path, index=False)


In [ ]:
import pandas as pd

# Read the CSV file
df = pd.read_csv('updated_merged_covar_condition_AD_R8.csv')

# Keep only the desired columns
df = df[['FID', 'SEX', 'age', 'age_at_onset2']]

# Replace non-finite values (NaN or inf) with -1
df['age_at_onset2'].fillna(-1, inplace=True)

# Convert 'age_at_onset2' column to integers
df['age_at_onset2'] = df['age_at_onset2'].astype(int)

# Save the modified DataFrame back to a CSV file
df.to_csv('updated2_merged_covar_condition_AD_R8.csv', index=False)


In [ ]:
import pandas as pd

# Read the CSV file
df = pd.read_csv('updated2_merged_covar_condition_AD_R8.csv')

# Rename the columns
df = df.rename(columns={'age': 'Age', 'age_at_onset2': 'Age_at_Onset'})

# Save the modified DataFrame back to a CSV file
df.to_csv('updated3_merged_covar_condition_AD_R8.csv', index=False)

In [ ]:
!gsutil -u $GOOGLE_PROJECT cat "gs://${v8_controlled_data_path}/wgs/short_read/snpindel/aux/ancestry/ancestry_preds.tsv" | cut -f1,4 > alldata_PCA_v8.tsv

In [ ]:
import pandas as pd

# Load the data
df = pd.read_csv('alldata_PCA_v8.tsv', sep='\t')

# Split the 'pca_features' column into 16 columns
df[['PCA1', 'PCA2', 'PCA3', 'PCA4', 'PCA5', 'PCA6', 'PCA7', 'PCA8', 'PCA9', 'PCA10', 'PCA11', 'PCA12', 'PCA13', 'PCA14', 'PCA15', 'PCA16']] = pd.DataFrame(df['pca_features'].apply(lambda x: x[1:-1].split(',')).tolist(), index=df.index)

# Drop the original 'pca_features' column
df.drop('pca_features', axis=1, inplace=True)

# Save the updated DataFrame
df.to_csv('alldata_PCA_v8_split.tsv', sep='\t', index=False)

In [ ]:
import pandas as pd

# Load the files
pca_data = pd.read_csv('alldata_PCA_v8_split.tsv', sep='\t')
covar_data = pd.read_csv('updated3_merged_covar_condition_AD_R8.csv')

# Merge the files
merged_data = pd.merge(pca_data, covar_data, left_on='research_id', right_on='FID', how='inner')

# Keep only the common columns
merged_data = merged_data[['research_id', 'FID', 'SEX', 'Age', 'Age_at_Onset', 'PCA1', 'PCA2', 'PCA3', 'PCA4', 'PCA5', 'PCA6', 'PCA7', 'PCA8', 'PCA9', 'PCA10', 'PCA11', 'PCA12', 'PCA13', 'PCA14', 'PCA15', 'PCA16']]

# Save the merged data to a new file 
merged_data.to_csv('covar_AD_FID_SEX_Age_AAO_PCA_data.csv', sep='\t', index=False)

# Rename columns
merged_data = merged_data.rename(columns={'research_id': 'FID', 'FID': 'IID'})

# Save the renamed merged data to a new file 
merged_data.to_csv('covar_AD_FID_IID_SEX_Age_AAO_PCA_data.csv', sep='\t', index=False)

In [ ]:
import pandas as pd

# Load the CSV file
df = pd.read_csv('covar_AD_FID_IID_SEX_Age_AAO_PCA_data.csv', sep='\t')

# Add a new column 'PHENO' with 2 values
df['PHENO'] = 2

# Save the updated DataFrame back to a CSV file
df.to_csv('covar_AD_FID_IID_SEX_Age_AAO_PCA_data_with_PHENO.csv', sep='\t', index=False)

In [ ]:
import pandas as pd

# Load the original CSV file
file_path = 'R8_controlperson_df.csv'
df = pd.read_csv(file_path)

# Select the desired columns
selected_columns = ['person_id', 'gender', 'date_of_birth']
new_df = df[selected_columns]

# Map gender values
def map_gender(gender):
    if gender == 'Male':
        return 1
    elif gender == 'Female':
        return 2
    else:
        return -9

new_df['gender'] = new_df['gender'].apply(map_gender)

# Calculate age from date_of_birth
new_df['date_of_birth'] = pd.to_datetime(new_df['date_of_birth'])
current_date = pd.to_datetime('today')
new_df['age'] = current_date.year - new_df['date_of_birth'].dt.year

# Rename the columns
new_df.rename(columns={
    'person_id': 'FID',
    'gender': 'SEX'
}, inplace=True)

# Save the new DataFrame to a new CSV file
new_file_path = 'covar_controls_R8_file.csv'
new_df.to_csv(new_file_path, index=False)


In [ ]:
import pandas as pd

# Load the data
df = pd.read_csv('covar_controls_R8_file.csv')

# Remove duplicate rows based on the 'FID' column
df = df.drop_duplicates(subset=['FID'])

# Save the updated DataFrame
df.to_csv('covar_controls_R8_uniqID_file.csv', index=False)

In [ ]:
# Load the files
control = pd.read_csv('R8_control_ids_new', header=None, names=['FID'])
covar = pd.read_csv('covar_controls_R8_uniqID_file.csv')

# Convert to string and strip whitespace
control['FID'] = control['FID'].astype(str).str.strip()
covar['FID'] = covar['FID'].astype(str).str.strip()

print("Number of IDs in control file:", len(control))
print("Number of IDs in covar file:", len(covar))

# Check intersection
intersection = set(control['FID']).intersection(set(covar['FID']))
print("Number of matching IDs:", len(intersection))

# Show first 10 matches
print("Example matches:", list(intersection)[:10])


In [ ]:
import pandas as pd

# Read control IDs as strings, ignore bad lines if any
controlsampleid_df = pd.read_csv(
    'R8_control_ids_new',
    header=None,
    names=['FID'],
    dtype=str
)

# Read covariates, make IDs strings
covar_df = pd.read_csv(
    'covar_controls_R8_uniqID_file.csv',
    dtype={'FID': str}
)

# Filter
filtered_covar_df = covar_df[covar_df['FID'].isin(controlsampleid_df['FID'])]

# Save
filtered_covar_df.to_csv('covar_control_R8_FID_SEX_Age_data.csv', index=False)


In [ ]:
import pandas as pd

# Load the CSV file
df = pd.read_csv('covar_control_R8_FID_SEX_Age_data.csv')

# Remove the date_of_birth column
df = df.drop(columns=['date_of_birth'])

# Rename the age column to Age
df = df.rename(columns={'age': 'Age'})

# Save the modified DataFrame to a new CSV file
df.to_csv('modified_covar_control_R8_FID_SEX_Age_data.csv', index=False)

In [ ]:
import pandas as pd

# Load the files
pca_data = pd.read_csv('alldata_PCA_v8_split.tsv', sep='\t')
covar_data = pd.read_csv('modified_covar_control_R8_FID_SEX_Age_data.csv')

# Merge the files
merged_data = pd.merge(pca_data, covar_data, left_on='research_id', right_on='FID', how='inner')

# Keep only the common columns
merged_data = merged_data[['research_id', 'FID', 'SEX', 'Age', 'PCA1', 'PCA2', 'PCA3', 'PCA4', 'PCA5', 'PCA6', 'PCA7', 'PCA8', 'PCA9', 'PCA10', 'PCA11', 'PCA12', 'PCA13', 'PCA14', 'PCA15', 'PCA16']]

# Save the merged data to a new file in tab-separated format
merged_data.to_csv('covar_control_R8_FID_SEX_Age_PCA_data.tsv', sep='\t', index=False)

# Rename columns
merged_data = merged_data.rename(columns={'research_id': 'FID', 'FID': 'IID'})

# Save the renamed merged data to a new file in tab-separated format
merged_data.to_csv('covar_control_R8_FID_IID_SEX_Age_PCA_data.tsv', sep='\t', index=False)


In [ ]:
import pandas as pd

# Read the original tab-separated file correctly
df = pd.read_csv('covar_control_R8_FID_IID_SEX_Age_PCA_data.tsv', sep='\t')

# Add PHENO column with value 1
df['PHENO'] = 1

# Save the DataFrame using tab (`\t`) as separator and enforce no quoting
df.to_csv('covar_control_R8_FID_IID_SEX_Age_PCA_data_with_PHENO.tsv',
          sep='\t', index=False, header=True, quoting=3)  

In [ ]:
import pandas as pd

# Load the file
df = pd.read_csv("covar_control_R8_FID_IID_SEX_Age_PCA_data_with_PHENO.tsv", sep='\t')

# Create the new column by copying the 'Age' column
df["Age_at_Onset"] = df["Age"]

# Reorder the columns as specified
column_order = [
    "FID", "IID", "SEX", "Age", "Age_at_Onset",
    "PCA1", "PCA2", "PCA3", "PCA4", "PCA5", "PCA6", "PCA7", "PCA8", "PCA9", "PCA10",
    "PCA11", "PCA12", "PCA13", "PCA14", "PCA15", "PCA16",
    "PHENO"
]

# Apply the column order
df = df[column_order]

# Save the modified file
df.to_csv("covar_control_R8_with_AgeAtOnset.tsv", sep='\t', index=False)

In [ ]:
df_cases = pd.read_csv("covar_AD_FID_IID_SEX_Age_AAO_PCA_data_with_PHENO.csv", sep='\t')

In [ ]:
import pandas as pd

# Read both files 
df_controls = pd.read_csv("covar_control_R8_with_AgeAtOnset.tsv", sep='\t')
df_cases = pd.read_csv("covar_AD_FID_IID_SEX_Age_AAO_PCA_data_with_PHENO.csv", sep='\t')

# remove duplicated header rows if they exist
if "FID" in df_cases.columns:
    df_cases = df_cases[df_cases["FID"] != "FID"]

# Merge controls and cases
df_merged = pd.concat([df_controls, df_cases], ignore_index=True)

# Save the merged output
df_merged.to_csv("covar_merged_R8_controls_ADcases.tsv", sep='\t', index=False)

In [ ]:
import pandas as pd

# Read burden sample list
burden = pd.read_csv("all_samples_burden.txt", header=None, names=["sample_id"], dtype=str)

# Read covariate file
covar = pd.read_csv("covar_merged_R8_controls_ADcases.tsv", sep="\t", dtype=str)

# Find overlapping samples 
overlap = covar[covar["IID"].isin(burden["sample_id"])]
print("Total number of samples in burden list:", len(burden))
print("Total overlapping samples:", len(overlap))

# Count by PHENO
print("\nCounts by PHENO value:")
print(overlap["PHENO"].value_counts())

# separate numbers:
num_cases = (overlap["PHENO"] == "2").sum()     
num_controls = (overlap["PHENO"] == "1").sum()  
print("\nNumber of cases:", num_cases)
print("Number of controls:", num_controls)


In [ ]:
import pandas as pd

# Load the sample list
samples = pd.read_csv("all_samples_burden.txt", header=0)
sample_ids = set(samples['sample_id'].astype(str))

# Function to filter and save new file
def filter_file(input_file, output_file, sep="\t"):
    df = pd.read_csv(input_file, sep=sep)
    # Keep only rows where IID is in sample_ids
    df_filtered = df[df['IID'].astype(str).isin(sample_ids)]
    df_filtered.to_csv(output_file, sep=sep, index=False)
    print(f"Saved filtered file: {output_file} ({len(df_filtered)} samples)")

# Filter the three files
filter_file("covar_control_R8_FID_IID_SEX_Age_PCA_data_with_PHENO.tsv", 
            "covar_control_R8_filtered.tsv", sep="\t")

filter_file("covar_AD_FID_IID_SEX_Age_AAO_PCA_data_with_PHENO.csv", 
            "covar_AD_R8_filtered.csv", sep="\t")

filter_file("covar_merged_R8_controls_ADcases.tsv", 
            "covar_merged_R8_filtered.tsv", sep="\t")


## Separate covar files by ancestry

In [ ]:
! gsutil cat gs://{bucket}/genotools_files/all_r8_results/all_r8_ancestry_umap_linearsvc_predicted_labels_merged.txt

In [ ]:
!gsutil cp gs://{bucket}/genotools_files/all_r8_results/all_r8_ancestry_umap_linearsvc_predicted_labels_merged.txt .

In [ ]:
!awk 'NR>1 {print $2"\t"$3}' all_r8_ancestry_umap_linearsvc_predicted_labels_merged.txt > iid_ancestry_map.txt

In [ ]:
import pandas as pd

# Load ancestry file

ancestry = pd.read_csv(
    "all_r8_ancestry_umap_linearsvc_predicted_labels_merged.txt",
    sep="\t",
    dtype=str
)

ancestry.columns = ancestry.columns.str.strip()

# Keep necessary columns
ancestry = ancestry[["IID", "label"]]

print("Ancestry loaded:", ancestry.shape)
print("Ancestry columns:", ancestry.columns.tolist())

# Function to split by ancestry

def split_by_ancestry(input_file, prefix):
    
    df = pd.read_csv(input_file, sep="\t", dtype=str)
    df.columns = df.columns.str.strip()
    
    print(f"\nProcessing: {input_file}")
    print("Original shape:", df.shape)

    # Merge ancestry (only IID + label)
    df = df.merge(ancestry, on="IID", how="inner")
    
    print("After merge:", df.shape)

    # Ensure FID is first column
    if "FID" in df.columns:
        cols = list(df.columns)
        cols.insert(0, cols.pop(cols.index("FID")))
        df = df[cols]

    # Split and save
    for anc in sorted(df["label"].unique()):
        df_sub = df[df["label"] == anc].drop(columns=["label"])
        output_name = f"{prefix}_{anc}.csv"
        df_sub.to_csv(output_name, index=False)
        print(f"Saved: {output_name}  (N={len(df_sub)})")

# Run for all three files

split_by_ancestry("covar_control_R8_filtered.tsv", "covar_control_R8_filtered")

split_by_ancestry("covar_AD_R8_filtered.csv", "covar_AD_R8_filtered")

split_by_ancestry("covar_merged_R8_filtered.tsv", "covar_merged_R8_filtered")


## Separate pheno files by ancestry

In [ ]:
import pandas as pd

# Load ancestry file

ancestry = pd.read_csv(
    "all_r8_ancestry_umap_linearsvc_predicted_labels_merged.txt",
    sep="\t",
    dtype=str
)

ancestry.columns = ancestry.columns.str.strip()
ancestry = ancestry[["IID", "label"]]

print("Ancestry loaded:", ancestry.shape)
print("Ancestry columns:", ancestry.columns.tolist())

# Function to keep only FID, IID, PHENO and split by ancestry

def split_pheno_by_ancestry(input_file, prefix):
    
    df = pd.read_csv(input_file, sep="\t", dtype=str)
    df.columns = df.columns.str.strip()
    
    print(f"\nProcessing: {input_file}")
    print("Original shape:", df.shape)
    
    # Merge ancestry
    df = df.merge(ancestry, on="IID", how="inner")
    print("After merge:", df.shape)
    
    # Keep only FID, IID, PHENO
    cols_to_keep = ["FID", "IID", "PHENO", "label"]
    df = df[cols_to_keep]
    
    # Ensure FID is first column
    cols = list(df.columns)
    cols.insert(0, cols.pop(cols.index("FID")))
    df = df[cols]
    
    # Split and save per ancestry
    for anc in sorted(df["label"].unique()):
        df_sub = df[df["label"] == anc].drop(columns=["label"])
        output_name = f"{prefix}_Pheno_{anc}.csv"
        df_sub.to_csv(output_name, index=False)
        print(f"Saved: {output_name}  (N={len(df_sub)})")

# Run for all three files

split_pheno_by_ancestry("covar_control_R8_filtered.tsv", "control_R8_filtered")
split_pheno_by_ancestry("covar_AD_R8_filtered.csv", "AD_R8_filtered")
split_pheno_by_ancestry("covar_merged_R8_filtered.tsv", "merged_R8_filtered")


## Create Plink files

In [ ]:
from datetime import datetime
import os
import pandas as pd
import hail as hl
import subprocess

# Start timing
start = datetime.now()

# Get workspace paths
bucket = os.getenv('WORKSPACE_BUCKET')
genomic_location = os.getenv("CDR_STORAGE_PATH")
vds_srwgs_path = os.getenv("WGS_VDS_PATH")

# Use default path if environment variable not set
if vds_srwgs_path is None:
    vds_srwgs_path = "gs://${v8_controlled_data_path}/wgs/short_read/snpindel/vds/hail.vds"

# Configuration parameters
MAF_THRESHOLDS = {
    "001": 0.01,
    "003": 0.03,
    "005": 0.05
}

VARIANT_TYPES = ["all", "coding"]

# Set Hail reference genome
hl.default_reference(new_default_reference="GRCh38")

In [ ]:

# Load AD cases from GCS (no header, one ID per line)
case_file = f"{bucket}/data/AD_R8_person_condition_df_uniqid"
print(f"Reading cases from: {case_file}")
result = subprocess.run(f"gsutil cat {case_file}", shell=True, capture_output=True, text=True, check=True)
case_ids = [line.strip() for line in result.stdout.split('\n') if line.strip()]
print(f"Loaded {len(case_ids)} AD cases")
print(f"First 5 case IDs: {case_ids[:5]}")

# Load controls from GCS (no header, one ID per line)
control_file = f"{bucket}/data/R8_control_ids"
print(f"\nReading controls from: {control_file}")
result = subprocess.run(f"gsutil cat {control_file}", shell=True, capture_output=True, text=True, check=True)
control_ids = [line.strip() for line in result.stdout.split('\n') if line.strip()]
print(f"Loaded {len(control_ids)} controls")
print(f"First 5 control IDs: {control_ids[:5]}")

# Combine all samples
all_sample_ids = case_ids + control_ids
print(f"\nTotal samples: {len(all_sample_ids)}")

# Save combined sample list for filtering
all_samples_df = pd.DataFrame({"sample_id": all_sample_ids})
all_samples_df.to_csv(f"{bucket}/data/all_samples_burden.txt", sep="\t", index=False)

In [ ]:

# Load full VDS
vds = hl.vds.read_vds(vds_srwgs_path)

test_intervals = [
    'chr12:120720774-120745008',
    'chr7:128022071-128037107',
    'chr14:74014349-74087863',
    'chr14:74079796-74206235',
    'chr19:43348686-43368172',
    'chr3:10244372-10290796',
    'chr3:10243023-10286218',
    'chr15:21646844-21657968',
    'chr6:125214962-125307078',
    'chr14:73605945-73624888',
    'chr11:62428542-62561235',
]

# Convert strings to Hail locus intervals
intervals = [hl.parse_locus_interval(x) for x in test_intervals]

# Filter VDS to only those intervals
vds = hl.vds.filter_intervals(vds, intervals)


In [ ]:
# Path to flagged samples
flagged_samples_path = "gs://${v8_controlled_data_path}/wgs/short_read/snpindel/aux/qc/flagged_samples.tsv"

# Import flagged samples table (distributed)
flagged_samples = hl.import_table(flagged_samples_path, key="s", min_partitions=200)
n_flagged = flagged_samples.count()
print(f"Loaded {n_flagged} flagged samples for removal")

# Filter VDS directly using Hail's distributed method
vds = hl.vds.filter_samples(vds, flagged_samples, keep=False, remove_dead_alleles=True)
print("Flagged samples removed from VDS")

# Count flagged cases and controls in a distributed way
case_ht = hl.Table.parallelize([{"s": sid} for sid in case_ids], key="s")
control_ht = hl.Table.parallelize([{"s": sid} for sid in control_ids], key="s")

flagged_cases_count = case_ht.join(flagged_samples, how="inner").count()
flagged_controls_count = control_ht.join(flagged_samples, how="inner").count()

case_ids_clean_ht = case_ht.anti_join(flagged_samples)
control_ids_clean_ht = control_ht.anti_join(flagged_samples)

case_ids_clean = list(case_ids_clean_ht.s.collect())
control_ids_clean = list(control_ids_clean_ht.s.collect())

all_sample_ids = case_ids_clean + control_ids_clean


In [ ]:

mt = hl.vds.to_dense_mt(vds)

In [ ]:
# Find missing sample

# Get actual samples in MT
mt_sample_ids = set(mt.s.collect())
print(f"Samples in MatrixTable: {len(mt_sample_ids)}")

# Check which of our samples are missing
expected_samples = set(all_sample_ids)
missing_samples = expected_samples - mt_sample_ids

print(f"\nExpected samples: {len(expected_samples)}")
print(f"Missing samples: {len(missing_samples)}")

if missing_samples:
    print("\nMissing sample ID(s):")
    for sid in missing_samples:
        if sid in case_ids_clean:
            sample_type = "CASE"
        else:
            sample_type = "CONTROL"
        print(f"  - {sid} ({sample_type})")
else:
    print("\nNo missing samples - all accounted for!")


In [ ]:
# Filtering to study samples
samples_to_keep = hl.import_table(f"{bucket}/data/all_samples_burden.txt", key="sample_id")
mt = mt.filter_cols(hl.is_defined(samples_to_keep[mt.s]))
n_samples = mt.count_cols()
print(f"Filtered to {n_samples} study samples")

In [ ]:
mt_plink = hl.split_multi_hts(mt)

In [ ]:
mt_plink = mt_plink.select_entries(
    GT = hl.unphased_diploid_gt_index_call(mt_plink.LGT.n_alt_alleles())
)

In [ ]:
out_path = f'{bucket}/data/meta_plink'

In [ ]:
hl.export_plink(mt_plink, out_path, ind_id = mt_plink.s)

## Separate plink files by ancestry

In [ ]:
import pandas as pd
import subprocess
import os

plink_prefix = "meta_plink"
my_bucket = os.getenv("WORKSPACE_BUCKET")

ancestries = [
    "AAC", "AFR", "AJ", "AMR", "CAH",
    "CAS", "EAS", "EUR", "SAS",
    "FIN", "MDE"
]

for anc in ancestries:
    print(f"\nProcessing {anc}")

    case_file = f"AD_R8_filtered_Pheno_{anc}.csv"
    control_file = f"control_R8_filtered_Pheno_{anc}.csv"

    # Check if files exist
    if not os.path.exists(case_file):
        print(f"WARNING: Case file missing for {anc}. Skipping.")
        continue

    if not os.path.exists(control_file):
        print(f"WARNING: Control file missing for {anc}. Skipping.")
        continue

    # Load files
    cases = pd.read_csv(case_file)
    controls = pd.read_csv(control_file)

    # Assign phenotype coding
    cases["PHENO"] = 2
    controls["PHENO"] = 1

    # Combine
    pheno = pd.concat([
        cases[["IID","PHENO"]],
        controls[["IID","PHENO"]]
    ])

    # Force FID = 0
    pheno["FID"] = 0

    # Save phenotype file
    pheno_file = f"pheno_{anc}.txt"
    pheno[["FID","IID","PHENO"]].to_csv(
        pheno_file,
        sep="\t",
        index=False,
        header=False
    )

    # Save keep file
    keep_file = f"keep_{anc}.txt"
    pheno[["FID","IID"]].to_csv(
        keep_file,
        sep="\t",
        index=False,
        header=False
    )

    # Run PLINK
    subprocess.run([
        "plink",
        "--bfile", plink_prefix,
        "--keep", keep_file,
        "--pheno", pheno_file,
        "--allow-no-sex",
        "--make-bed",
        "--out", f"meta_plink_{anc}"
    ])

    # Upload
    subprocess.run([
        "gsutil", "-m", "cp",
        f"meta_plink_{anc}.bed",
        f"meta_plink_{anc}.bim",
        f"meta_plink_{anc}.fam",
        f"{my_bucket}/ancestry_split/"
    ])


In [ ]:
import pandas as pd
import os

plink_prefix = "meta_plink"

ancestries = [
    "AAC", "AFR", "AJ", "AMR", "CAH",
    "CAS", "EAS", "EUR", "SAS",
    "FIN", "MDE"
]

print("Ancestry | #Cases_in_FAM | #Controls_in_FAM | Total_in_FAM")

for anc in ancestries:
    fam_file = f"meta_plink_{anc}.fam"  
    case_file = f"AD_R8_filtered_Pheno_{anc}.csv"
    control_file = f"control_R8_filtered_Pheno_{anc}.csv"

    # Skip if files don't exist
    if not os.path.exists(fam_file):
        print(f"{anc} | FAM missing")
        continue
    if not os.path.exists(case_file):
        print(f"{anc} | Case file missing")
        continue
    if not os.path.exists(control_file):
        print(f"{anc} | Control file missing")
        continue

    # Load FAM (columns: FID IID PID MID SEX PHENO)
    fam = pd.read_csv(fam_file, sep=r"\s+", header=None)
    fam.columns = ["FID", "IID", "PID", "MID", "SEX", "PHENO"]

    # Load case/control CSVs
    cases = pd.read_csv(case_file)
    controls = pd.read_csv(control_file)

    # Keep only samples that are in FAM
    n_cases = cases[cases["IID"].isin(fam["IID"])].shape[0]
    n_controls = controls[controls["IID"].isin(fam["IID"])].shape[0]
    total = fam.shape[0]

    print(f"{anc} | {n_cases} | {n_controls} | {total}")


## Prepare files for REGENIE

In [ ]:
%%bash
set -euo pipefail

ancestries="EUR"

for anc in $ancestries
do
    echo "Processing ancestry: ${anc}"

    infile="meta_plink_${anc}"
    PFX="meta_${anc}_raw"

    # Convert PLINK1 to PLINK2
    plink2 \
      --bfile ${infile} \
      --make-pgen \
      --out ${PFX}

    # Identify duplicated positions (CHR:POS)
    awk 'NR>1 {key=$1":"$2; c[key]++} END{for(k in c) if(c[k]>1) print k}' \
      ${PFX}.pvar > dup.pos

    # Keep only biallelic SNPs (ACGT) and unique positions
    awk 'BEGIN{FS=OFS="\t"}
         NR==FNR {dup[$1]=1; next}
         NR==1 {next}
         {
           key=$1":"$2
           is_snp=(length($4)==1 && length($5)==1 && $4~/^[ACGT]$/ && $5~/^[ACGT]$/)
           if(is_snp && !(key in dup)) print $3
         }' dup.pos ${PFX}.pvar > keep.ids

    # QC filtering
    plink2 \
      --pfile ${PFX} \
      --extract keep.ids \
      --geno 0.05 \
      --hwe 1e-6 midp \
      --make-pgen \
      --out meta_${anc}_clean

    # MAF thresholds
    for maf in 0.01 0.03 0.05
    do
        maf_tag=$(echo $maf | sed 's/0\.//')
        plink2 \
          --pfile meta_${anc}_clean \
          --max-maf ${maf} \
          --mac 1 \
          --make-pgen \
          --out meta_${anc}_MAF${maf_tag}

        # Upload to bucket
        gsutil -m cp meta_${anc}_MAF${maf_tag}.* $WORKSPACE_BUCKET/ancestry_split/
    done

    echo "done: Ancestry ${anc} completed and uploaded"
done

In [ ]:
%%bash
set -euo pipefail

# Ancestry list and MAF codes
ancestries="EUR"
maf_codes="01 03 05"

echo " Extract coding variants (All of Us)"

# Step 1: Create CDS BED file 
if [ ! -f cds_hg38.bed ]; then

    if [ ! -f gencode.v38.annotation.gtf.gz ]; then
        echo "Downloading GENCODE v38 annotation..."
        wget -q https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_38/gencode.v38.annotation.gtf.gz
    fi

    echo "Creating CDS BED file..."
    zcat gencode.v38.annotation.gtf.gz \
        | awk '$3=="CDS"' \
        | awk -v OFS="\t" '{print $1, $4-1, $5}' \
        | sort -k1,1 -k2,2n \
        | bedtools merge -i - \
        > cds_hg38.bed

    echo "done: CDS BED created with $(wc -l < cds_hg38.bed) regions"
    echo ""
fi

# Step 2: Loop over ancestries and MAFs
for anc in $ancestries; do

    echo " Processing ancestry: ${anc}"

    for maf in $maf_codes; do

        infile="meta_${anc}_MAF${maf}"

        echo "---- MAF ${maf} ----"

        if [ ! -f "${infile}.pgen" ]; then
            echo "WARNING: Missing file: ${infile}.pgen"
            continue
        fi

        # Extract coding variants
        plink2 \
            --pfile ${infile} \
            --extract bed1 cds_hg38.bed \
            --make-pgen \
            --out ${infile}_coding

        # Variant counts
        total=$(tail -n +2 ${infile}.pvar | wc -l)
        coding=$(tail -n +2 ${infile}_coding.pvar | wc -l)

        if [ "$total" -gt 0 ]; then
            pct=$(python3 -c "print(round($coding*100/$total,2))")
        else
            pct=0
        fi

        echo "done: Coding variants: $coding / $total (${pct}%)"
        echo ""

        # Upload coding files to bucket
        gsutil -m cp ${infile}_coding.* $WORKSPACE_BUCKET/ancestry_split/

        echo "done: Uploaded ${infile}_coding files to bucket"
        echo ""
    done

    echo "done: Completed ancestry: ${anc}"
    echo ""
done


In [ ]:
%%bash
set -euo pipefail

# Settings
ancestries="EUR"
mafs="01 03 05"
OUTDIR="$WORKSPACE_BUCKET/ancestry_split"

echo " Creating gene-based annotations"

# Step 1: Create protein-coding gene BED (hg38)

# Recreate if missing OR empty
if [ ! -s genes_hg38.bed ]; then

    echo "Creating genes_hg38.bed..."

    if [ ! -f gencode.v38.annotation.gtf.gz ]; then
        echo "Downloading GENCODE v38..."
        wget -q https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_38/gencode.v38.annotation.gtf.gz
    fi

    zcat gencode.v38.annotation.gtf.gz | \
    awk -F'\t' '
    $3=="gene" && ($9 ~ /gene_type "protein_coding"/ || $9 ~ /gene_biotype "protein_coding"/) {
        split($9,a,"gene_name \"")
        split(a[2],b,"\"")
        gene=b[1]
        print $1"\t"$4-1"\t"$5"\t"gene
    }' > genes_hg38.bed

    echo "done: $(wc -l < genes_hg38.bed) protein-coding genes extracted"
    echo ""
fi

# Quick sanity check
if [ ! -s genes_hg38.bed ]; then
    echo "ERROR: genes_hg38.bed is empty. Stopping."
    exit 1
fi

# Upload gene BED
gsutil -m cp genes_hg38.bed $OUTDIR/
echo "done: Uploaded genes_hg38.bed to bucket"
echo ""

# Step 2: Loop over ancestries / MAF / variant types
for anc in $ancestries; do

    echo " Processing ancestry: ${anc}"

    for maf in $mafs; do
        for vtype in all coding; do

            if [ "$vtype" == "all" ]; then
                pfx="meta_${anc}_MAF${maf}"
            else
                pfx="meta_${anc}_MAF${maf}_coding"
            fi

            echo ""
            echo "----------- ${pfx} -----------"

            if [ ! -f "${pfx}.pvar" ]; then
                echo "WARNING: Missing file: ${pfx}.pvar — skipping"
                continue
            fi

            # Create variant BED 
            awk 'NR>1 {
                chr=$1
                if (chr !~ /^chr/) chr="chr"chr
                print chr"\t"($2-1)"\t"$2"\t"$3"\t"$4"\t"$5
            }' ${pfx}.pvar > ${pfx}_variants.bed

            echo "Mapping variants to genes..."

            bedtools intersect \
                -a ${pfx}_variants.bed \
                -b genes_hg38.bed \
                -wa -wb \
                | awk '{print $1"\t"($2+1)"\t"$4"\t"$5"\t"$9}' \
                > ${pfx}_anno.txt

            # Gene sets file
            cut -f5 ${pfx}_anno.txt | sort -u > ${pfx}_sets.txt

            # REGENIE mask file
            
            cat > ${pfx}_mask.txt << EOF
# REGENIE mask definition file
# Format: MASK_NAME CONSEQUENCE_TYPES
LOF stop_gained,frameshift_variant,splice_acceptor_variant,splice_donor_variant
MISSENSE missense_variant
DAMAGING stop_gained,frameshift_variant,splice_acceptor_variant,splice_donor_variant,missense_variant
EOF

            echo "done: $(wc -l < ${pfx}_anno.txt) variant–gene annotations"
            echo "done: $(wc -l < ${pfx}_sets.txt) gene sets"

            # Upload outputs to bucket
            gsutil -m cp \
                ${pfx}_anno.txt \
                ${pfx}_sets.txt \
                ${pfx}_mask.txt \
                ${pfx}_variants.bed \
                $OUTDIR/

            echo "done: Uploaded ${pfx} outputs to bucket"
            echo ""

        done
    done

    echo "done: Completed ancestry: ${anc}"
    echo ""
done


## Extract Common Variants for REGENIE Step 1 - using array data

In [ ]:
!gsutil -u $GOOGLE_PROJECT -m cp gs://${v8_controlled_data_path}/microarray/plink/* .

In [ ]:
%%bash
set -euo pipefail

# Input files
ARRAYS_PREFIX="arrays"           
EUR_FAM="meta_plink_EUR.fam"    
OUTPUT_PREFIX="arrays_EUR"       

# Step 1: Create a list of IID/FID to keep

awk '{print $1, $2}' $EUR_FAM > keep_EUR.txt
echo "done: Created keep list with $(wc -l < keep_EUR.txt) samples"

# Step 2: Subset the arrays dataset using PLINK2
plink2 \
    --bfile $ARRAYS_PREFIX \
    --keep keep_EUR.txt \
    --make-pgen \
    --out $OUTPUT_PREFIX

echo "done: Subsetted arrays dataset to EUR samples"
echo "Output files:"
echo "  ${OUTPUT_PREFIX}.pgen"
echo "  ${OUTPUT_PREFIX}.pvar"
echo "  ${OUTPUT_PREFIX}.psam"

# Step 3: Upload to bucket 

OUTDIR="$WORKSPACE_BUCKET/ancestry_split"

gsutil -m cp \
    ${OUTPUT_PREFIX}.pgen \
    ${OUTPUT_PREFIX}.pvar \
    ${OUTPUT_PREFIX}.psam \
    $OUTDIR/

echo "done: Uploaded ${OUTPUT_PREFIX} files to $OUTDIR"


In [ ]:
%%bash
set -euo pipefail

# Ancestry list 
ancestries="EUR"

# Output bucket path 

OUTDIR="$WORKSPACE_BUCKET/ancestry_split"

if [ -z "$OUTDIR" ]; then
    echo "ERROR: Please set the WORKSPACE_BUCKET environment variable"
    exit 1
fi


for anc in $ancestries; do
    echo "--------------------------------------------"
    echo "Processing ancestry: ${anc}"
    echo "Input: arrays_${anc}.pgen/pvar/psam"
    echo "--------------------------------------------"

    # Input prefix
    INPUT="arrays_${anc}"

    # 1) Extract COMMON variants
    plink2 \
        --pfile ${INPUT} \
        --maf 0.05 \
        --mac 20 \
        --geno 0.05 \
        --hwe 1e-6 \
        --make-pgen \
        --out array_${anc}_common

    echo "  done: Extracted common variants"

    # 2) LD pruning
    plink2 \
        --pfile array_${anc}_common \
        --indep-pairwise 1000 100 0.5 \
        --out array_${anc}_common_pruned

    # Safety check
    if [ ! -f array_${anc}_common_pruned.prune.in ]; then
        echo "  ERROR: ERROR: No prune.in file created for ${anc}"
        exit 1
    fi

    prune_count=$(wc -l < array_${anc}_common_pruned.prune.in)
    echo "  done: LD-pruned variants: ${prune_count}"

    # 3) Extract pruned variants
    plink2 \
        --pfile array_${anc}_common \
        --extract array_${anc}_common_pruned.prune.in \
        --make-pgen \
        --out array_${anc}_step1_variants

    final_count=$(tail -n +2 array_${anc}_step1_variants.pvar | wc -l)
    echo "  done: Final Step 1 variants: ${final_count}"

    # Upload outputs to bucket
    echo "  Uploading Step 1 outputs to bucket..."
    gsutil -m cp \
        array_${anc}_step1_variants.pgen \
        array_${anc}_step1_variants.pvar \
        array_${anc}_step1_variants.psam \
        $OUTDIR/

    echo "  done: Uploaded Step 1 outputs to $OUTDIR"
    echo ""
done


## Separate chr 1-22 for further analysis

In [ ]:
%%bash
set -euo pipefail

ancestries="EUR"

for anc in $ancestries; do
    echo "Filtering ${anc} to autosomes..."

    plink2 \
      --pfile array_${anc}_step1_variants \
      --chr 1-22 \
      --make-pgen \
      --out array_${anc}_step1_variants.autosomes
done


In [ ]:
import subprocess

# All ancestries
ancestries = ["EUR"]

# MAF groups
maf_groups = ["MAF01", "MAF03", "MAF05"]

# File types
file_types = ["", "_coding"]   # "" = all variants, "_coding" = coding only

for anc in ancestries:
    for maf in maf_groups:
        for ftype in file_types:

            input_prefix = f"meta_{anc}_{maf}{ftype}"
            output_prefix = f"{input_prefix}.autosomes"

            print(f"Filtering {input_prefix} to autosomes...")

            cmd = [
                "plink2",
                "--pfile", input_prefix,
                "--chr", "1-22",
                "--make-pgen",
                "--out", output_prefix
            ]

            subprocess.run(cmd, check=True)


## Correct input format and using REGENIE 

In [ ]:
%%bash
set -e

# 1. Create local bin directory
mkdir -p ~/bin
cd ~/bin

# 2. Clean old files
rm -f regenie regenie_v3.2.9.gz_x86_64_Linux.zip

# 3. Download regenie
wget -q https://github.com/rgcgithub/regenie/releases/download/v3.2.9/regenie_v3.2.9.gz_x86_64_Linux.zip

# 4. Unzip
unzip -o regenie_v3.2.9.gz_x86_64_Linux.zip

# 5. Rename + permissions
mv regenie_v3.2.9.gz_x86_64_Linux regenie
chmod +x regenie

# 6. Test directly 
./regenie --help


In [ ]:
# Rebuild psam for REGENIE
!awk 'NR>1 {print "0",$1,"0 0 0 -9"}' array_EUR_step1_variants.autosomes.psam \
    | cat <(echo "#FID IID PAT MAT SEX PHENO") - \
    > array_EUR_step1_variants.autosomes.psam.fixed

# Replace original file 
!mv array_EUR_step1_variants.autosomes.psam.fixed array_EUR_step1_variants.autosomes.psam


In [ ]:
! awk 'NR==1 {print "FID\tIID\tPHENO"} NR>1 {print "0\t" $2 "\t" $3}' \
  merged_R8_filtered_Pheno_EUR.txt > merged_R8_filtered_Pheno_EUR_fixed.txt

In [ ]:
import pandas as pd

# Files
psam_file = "array_EUR_step1_variants.autosomes.psam"
pheno_file = "merged_R8_filtered_Pheno_EUR.txt"
psam_fixed = psam_file + ".fixed"

# Load psam, remove '#' from header
psam = pd.read_csv(psam_file, delim_whitespace=True, comment=None)
psam.columns = [c.lstrip("#").strip().upper() for c in psam.columns]

# Load phenotype file
pheno = pd.read_csv(pheno_file, delim_whitespace=True)
pheno.columns = pheno.columns.str.strip().str.upper()

# Update PHENO from phenotype file (keep FID = 0)
psam["PHENO"] = psam["IID"].map(pheno.set_index("IID")["PHENO"])

# Check missing PHENO
missing = psam['PHENO'].isna().sum()
print(f"Number of individuals with missing PHENO: {missing}")

# Save updated psam without index
psam.to_csv(psam_fixed, sep="\t", index=False)

# Add '#' to the header line 
with open(psam_fixed, "r") as f:
    lines = f.readlines()

lines[0] = "#" + lines[0]  

with open(psam_fixed, "w") as f:
    f.writelines(lines)

# Replace original psam
import os
os.replace(psam_fixed, psam_file)


In [ ]:
import pandas as pd

psam = pd.read_csv("array_EUR_step1_variants.autosomes.psam", delim_whitespace=True, comment=None)
psam.columns = [c.lstrip("#").strip().upper() for c in psam.columns]

pheno = pd.read_csv("merged_R8_filtered_Pheno_EUR.txt", delim_whitespace=True)
pheno.columns = pheno.columns.str.strip().str.upper()

psam['IID'] = psam['IID'].astype(str)
pheno['IID'] = pheno['IID'].astype(str)

matched = psam['IID'].isin(pheno['IID']).sum()
total = len(psam)
print(f"{matched}/{total} individuals in psam have matching PHENO")


In [ ]:
%%bash
awk 'BEGIN {OFS="\t"} 
NR==1 {
    printf "FID\tIID"
    for(i=3; i<=NF; i++) printf "\t%s", $i
    printf "\n"
} 
NR>1 {
    printf "0\t%s", $2
    for(i=3; i<=NF; i++) printf "\t%s", $i
    printf "\n"
}' covar_merged_R8_filtered_EUR.txt > covar_merged_R8_filtered_EUR_fixed.txt

## REGENIE Step 1 - Fit Null Model

In [ ]:
#!/usr/bin/env python3

import subprocess

# Settings
ancestries = ["EUR"]  
OUTDIR = "$WORKSPACE_BUCKET/ancestry_split"  

for anc in ancestries:
    PGEN = f"array_{anc}_step1_variants.autosomes"
    PHENO = f"merged_R8_filtered_Pheno_{anc}_fixed.txt"
    COVAR = f"covar_merged_R8_filtered_{anc}_fixed.txt"

    # Bash commands for this ancestry
   
    bash_commands = f"""
set -euo pipefail

echo " REGENIE Step 1 - {anc}"

echo "Checking input files..."
ls -lh {PGEN}.pgen {PGEN}.pvar {PGEN}.psam
ls -lh {PHENO}
ls -lh {COVAR}

VAR_COUNT=$(tail -n +2 {PGEN}.pvar | wc -l)
echo "Variant count: $VAR_COUNT"

if [ "$VAR_COUNT" -lt 1000 ]; then
    echo "ERROR: Too few variants (<1000)"
    exit 1
fi

~/bin/regenie \\
  --step 1 \\
  --pgen {PGEN} \\
  --phenoFile {PHENO} \\
  --covarFile {COVAR} \\
  --phenoCol PHENO \\
  --covarColList SEX,Age_at_Onset,PCA1,PCA2,PCA3,PCA4,PCA5,PCA6,PCA7,PCA8,PCA9,PCA10 \\
  --bt \\
  --cc12 \\
  --bsize 1000 \\
  --threads 8 \\
  --lowmem \\
  --lowmem-prefix tmp_{anc} \\
  --force-step1 \\
  --gz \\
  --out array_step1_{anc}

echo "Checking LOCO files..."
ls array_step1_{anc}_*.loco.gz | wc -l || true
echo "Step 1 done."


if [ -z "$OUTDIR" ]; then
    echo "ERROR: Please set the WORKSPACE_BUCKET environment variable"
    exit 1
fi

echo "Uploading Step 1 outputs to $OUTDIR..."

# Upload main genotype files
gsutil -m cp \\
    array_step1_{anc}.pgen \\
    array_step1_{anc}.pvar \\
    array_step1_{anc}.psam \\
    $OUTDIR/

# Upload additional outputs (*.loco.gz, *.bt.gz, *.log.gz)
gsutil -m cp array_step1_{anc}_*.loco.gz $OUTDIR/ || true
gsutil -m cp array_step1_{anc}_*.bt.gz $OUTDIR/ || true
gsutil -m cp array_step1_{anc}_*.log.gz $OUTDIR/ || true

echo "done: Uploaded Step 1 outputs for {anc} to $OUTDIR"
echo ""
"""

    # Run the Bash commands directly
    subprocess.run(bash_commands, shell=True, check=True, executable="/bin/bash")


In [ ]:
%%bash
OUTDIR="$WORKSPACE_BUCKET/ancestry_split"
gsutil -m cp array_step1_EUR_1.loco.gz $OUTDIR/
gsutil -m cp array_step1_EUR_pred.list $OUTDIR/
gsutil -m cp array_step1_EUR.log $OUTDIR/

## Create files for Step 2 - REGENIE

In [ ]:
%%bash
set -euo pipefail

# Ancestry list
ancestries=("EUR")

# MAF thresholds
mafs=("01" "03" "05")

# Gene BED 
GENES_BED="genes_hg38.bed"

# Bucket output directory
OUTDIR="$WORKSPACE_BUCKET/ancestry_split"

for anc in "${ancestries[@]}"; do
    echo "======================================="
    echo "  Processing ancestry: $anc"
    echo "======================================="

    for maf in "${mafs[@]}"; do
        for vtype in all coding; do

            if [ "$vtype" == "all" ]; then
                pfx="meta_${anc}_MAF${maf}.autosomes"
            else
                pfx="meta_${anc}_MAF${maf}_coding.autosomes"
            fi

            echo ""
            echo "----------- ${pfx} -----------"

            if [ ! -f "${pfx}.pvar" ]; then
                echo "WARNING: File missing: ${pfx}.pvar - skipping"
                continue
            fi

            # Create variant BED 
            awk 'NR>1 {
                chr=$1
                if (chr !~ /^chr/) chr="chr"chr
                print chr"\t"($2-1)"\t"$2"\t"$3"\t"$4"\t"$5
            }' ${pfx}.pvar > ${pfx}_variants.bed

            # Map variants to genes
            bedtools intersect \
                -a ${pfx}_variants.bed \
                -b ${GENES_BED} \
                -wa -wb \
                | awk '{print $1"\t"($2+1)"\t"$4"\t"$5"\t"$9}' \
                > ${pfx}_anno.txt

            # Create gene sets
            cut -f5 ${pfx}_anno.txt | sort -u > ${pfx}_sets.txt

            # Create REGENIE mask file
            cat > ${pfx}_mask.txt << 'EOF'
# REGENIE mask definition file
# Format: MASK_NAME CONSEQUENCE_TYPES
LOF stop_gained,frameshift_variant,splice_acceptor_variant,splice_donor_variant
MISSENSE missense_variant
DAMAGING stop_gained,frameshift_variant,splice_acceptor_variant,splice_donor_variant,missense_variant
EOF

            echo "done: $(wc -l < ${pfx}_anno.txt) annotations created"
            echo "done: $(wc -l < ${pfx}_sets.txt) gene sets created"
            echo "done: Mask file ${pfx}_mask.txt created"

            # Upload outputs to bucket
            gsutil -m cp \
                ${pfx}_variants.bed \
                ${pfx}_anno.txt \
                ${pfx}_sets.txt \
                ${pfx}_mask.txt \
                ${OUTDIR}/

            echo "done: Uploaded ${pfx} outputs to bucket"
            echo ""

        done
    done

    echo "======================================="
    echo "  Completed ancestry: $anc"
    echo "======================================="
done


In [ ]:
%%bash
set -euo pipefail

# Ancestry list
ancestries=("EUR")

# MAF thresholds
mafs=("01" "03" "05")

# Path to genes BED file
GENES_BED="genes_hg38.bed"  

# Bucket output directory
OUTDIR="$WORKSPACE_BUCKET/ancestry_split"

for anc in "${ancestries[@]}"; do
    echo "  Processing ancestry: $anc"

    for maf in "${mafs[@]}"; do
        for vtype in all coding; do

            # Define input prefix
            if [ "$vtype" == "all" ]; then
                pfx="meta_${anc}_MAF${maf}.autosomes"
            else
                pfx="meta_${anc}_MAF${maf}_coding.autosomes"
            fi

            echo ""
            echo "----------- ${pfx} -----------"

            # Skip if .pvar missing
            if [ ! -f "${pfx}.pvar" ]; then
                echo "WARNING: File missing: ${pfx}.pvar — skipping"
                continue
            fi

            # Variant BED file 
            awk 'NR>1 {
                chr=$1
                if (chr !~ /^chr/) chr="chr"chr
                print chr"\t"($2-1)"\t"$2"\t"$3"\t"$4"\t"$5
            }' ${pfx}.pvar > ${pfx}_variants_gene.bed

            # Map variants to genes
            bedtools intersect \
                -a ${pfx}_variants_gene.bed \
                -b ${GENES_BED} \
                -wa -wb \
            | awk '{
                chr=$1;
                pos=$3;      
                ref=$4;
                alt=$5;
                gene=$10;    
                print chr"\t"pos"\t"ref"\t"alt"\t"gene
            }' \
            > ${pfx}_anno_gene.txt

            # Create gene set file
            cut -f5 ${pfx}_anno_gene.txt | sort -u > ${pfx}_sets_gene.txt

            # Create mask file
            cat > ${pfx}_mask_gene.txt << 'EOF'
# REGENIE mask definition file
# Format: MASK_NAME CONSEQUENCE_TYPES
LOF stop_gained,frameshift_variant,splice_acceptor_variant,splice_donor_variant
MISSENSE missense_variant
DAMAGING stop_gained,frameshift_variant,splice_acceptor_variant,splice_donor_variant,missense_variant
EOF

            echo "done: $(wc -l < ${pfx}_anno_gene.txt) annotations created"
            echo "done: $(wc -l < ${pfx}_sets_gene.txt) gene sets created"
            echo "done: Mask file ${pfx}_mask_gene.txt created"

            # Upload outputs to bucket
            gsutil -m cp \
                ${pfx}_variants_gene.bed \
                ${pfx}_anno_gene.txt \
                ${pfx}_sets_gene.txt \
                ${pfx}_mask_gene.txt \
                ${OUTDIR}/

            echo "done: Uploaded ${pfx} outputs to bucket"
            echo ""

        done
    done

    echo "======================================="
    echo "  Completed ancestry: $anc"
    echo "======================================="
done


In [ ]:
#!/usr/bin/env python3

import os
import subprocess

# SETTINGS
ANCESTRIES = ["EUR"]
MAF_CODES = ["01", "03", "05"]
VTYPES = ["", "_coding"]   

# Bucket output directory
OUTDIR = os.environ.get("WORKSPACE_BUCKET", "") + "/ancestry_split"

if not OUTDIR:
    raise ValueError("Please set $WORKSPACE_BUCKET environment variable!")

# Helper
def read_lines(path):
    with open(path, "r") as f:
        return [l.strip() for l in f if l.strip()]

def upload_to_bucket(files):
    """Uploads a list of local files to OUTDIR in the bucket"""
    for f in files:
        if os.path.exists(f):
            cmd = ["gsutil", "cp", f, OUTDIR + "/"]
            subprocess.run(cmd, check=True)
            print(f"     done: Uploaded {f} to bucket")
        else:
            print(f"     WARNING: File {f} does not exist, skipping upload")

# Main Loop
for anc in ANCESTRIES:
    print(f"\n\n### Processing ancestry: {anc}")
    for maf in MAF_CODES:
        for vt in VTYPES:

            vt_name = "all" if vt == "" else "coding"
            prefix = f"meta_{anc}_MAF{maf}{vt}.autosomes"

            anno_in = f"{prefix}_anno_gene.txt"
            sets_in = f"{prefix}_sets_gene.txt"
            mask_in = f"{prefix}_mask_gene.txt"

            # --- Check files exist ---
            if not os.path.exists(anno_in):
                print(f"  ERROR: Skipping {prefix} (missing {anno_in})")
                continue

            print(f"\n  → {prefix}")
            print("     Input:", anno_in)

            # Part 1: Build REGENIE-style annotation (variant . gene)
            anno_regenie = f"{prefix}_anno_regenie.txt"
            with open(anno_in, "r") as fin, open(anno_regenie, "w") as fout:
                for line in fin:
                    parts = line.strip().split()
                    var = parts[2]   # 1-based POS
                    gene = parts[-1]
                    fout.write(f"{var}\t.\t{gene}\n")
            print("     done: Built:", anno_regenie)

            # Part 2: Build M1 annotation
            anno_m1 = f"{prefix}_anno_M1.txt"
            with open(anno_regenie, "r") as fin, open(anno_m1, "w") as fout:
                for line in fin:
                    var, _, gene = line.strip().split()
                    fout.write(f"{var}\tM1\t{gene}\n")
            print("     done: Built:", anno_m1)

            # Part 3: Convert to proper 4-column format for REGENIE
            anno_proper = f"{prefix}_anno_proper4.txt"
            with open(anno_m1, "r") as fin, open(anno_proper, "w") as fout:
                for line in fin:
                    var, _, gene = line.strip().split()
                    fout.write(f"{var}\t{gene}\t.\tALL\n")
            print("     done: Built:", anno_proper)

            # Part 4: Build set-list (GENE  CHR  START  VAR1,VAR2,...)
            
            setdict = {}   
            with open(anno_m1, "r") as fin:
                for line in fin:
                    var, _, gene = line.strip().split()
                    chrom = var.split(":")[0]
                    pos = int(var.split(":")[1])
                    if gene not in setdict:
                        setdict[gene] = {"chr": chrom, "start": pos, "vars": [var]}
                    else:
                        setdict[gene]["vars"].append(var)
                        if pos < setdict[gene]["start"]:
                            setdict[gene]["start"] = pos

            sets_proper = f"{prefix}_sets_proper4.txt"
            with open(sets_proper, "w") as fout:
                for gene in sorted(setdict.keys()):
                    chrom = setdict[gene]["chr"]
                    start = setdict[gene]["start"]
                    varlist = ",".join(setdict[gene]["vars"])
                    fout.write(f"{gene}\t{chrom}\t{start}\t{varlist}\n")
            print("     done: Built:", sets_proper)

            # Part 5: Build Mask 
            mask_proper = f"{prefix}_mask_proper4.txt"
            with open(mask_proper, "w") as fout:
                fout.write("M1\tALL\n")
            print("     done: Built:", mask_proper)

            # Upload all outputs to bucket
            
            upload_to_bucket([anno_regenie, anno_m1, anno_proper, sets_proper, mask_proper])


In [ ]:
# Fix psam file
%%bash
set -euo pipefail

maf_codes="01 03 05"

for maf in $maf_codes; do
    echo "Processing MAF${maf}..."

    # ---------- Regular ----------
    awk 'NR>1 {print "0",$1,"0 0 0 -9"}' meta_EUR_MAF${maf}.autosomes.psam \
        | cat <(echo "#FID IID PAT MAT SEX PHENO") - \
        > meta_EUR_MAF${maf}.autosomes_fixed.psam

    # ---------- Coding ----------
    awk 'NR>1 {print "0",$1,"0 0 0 -9"}' meta_EUR_MAF${maf}_coding.autosomes.psam \
        | cat <(echo "#FID IID PAT MAT SEX PHENO") - \
        > meta_EUR_MAF${maf}_coding.autosomes_fixed.psam

done

echo "All psam files rebuilt successfully."


In [ ]:
import pandas as pd
import os

pheno_file = "merged_R8_filtered_Pheno_EUR.txt"

# Load phenotype 
pheno = pd.read_csv(pheno_file, delim_whitespace=True)
pheno.columns = pheno.columns.str.strip().str.upper()
pheno_map = pheno.set_index("IID")["PHENO"]

maf_codes = ["01", "03", "05"]

for maf in maf_codes:
    
    for suffix in ["", "_coding"]:
        
        psam_file = f"meta_EUR_MAF{maf}{suffix}.autosomes_fixed.psam"
        
        print(f"\nProcessing {psam_file}...")
        
        psam = pd.read_csv(psam_file, delim_whitespace=True)
        psam.columns = [c.lstrip("#").strip().upper() for c in psam.columns]
        
        psam["PHENO"] = psam["IID"].map(pheno_map)
        
        missing = psam["PHENO"].isna().sum()
        print(f"Missing PHENO: {missing}")
        
        # Save back to same file
        psam.to_csv(psam_file, sep="\t", index=False)
        
        # Add '#' to header
        with open(psam_file, "r") as f:
            lines = f.readlines()
        lines[0] = "#" + lines[0]
        with open(psam_file, "w") as f:
            f.writelines(lines)
        
        print(f"Updated: {psam_file}")


In [ ]:
%%bash
set -euo pipefail

maf_codes="01 03 05"

for maf in $maf_codes; do
    for suffix in "" "_coding"; do
        
        old="meta_EUR_MAF${maf}${suffix}.autosomes.psam"
        new="meta_EUR_MAF${maf}${suffix}.autosomes_fixed.psam"
        
        echo "Replacing $old with $new"
        
        mv "$new" "$old"
        
    done
done

echo "All _fixed.psam files renamed to original .psam"


## REGENIE Step 2

In [ ]:
#!/usr/bin/env python3

import os
import subprocess

# ---------- CONFIG ----------
ANCESTRIES = ["EUR"]
MAF_CODES = ["01", "03", "05"]
VAR_TYPES = ["", "coding"]
THREADS = 8  
OUTDIR_BASE = os.environ.get("WORKSPACE_BUCKET", "") + "/regenie_step2"

for anc in ANCESTRIES:
    pheno_file = f"merged_R8_filtered_Pheno_{anc}_fixed.txt"
    covar_file = f"covar_merged_R8_filtered_{anc}_fixed.txt"
    pred_file = f"array_step1_{anc}_pred.list"
    
    for maf in MAF_CODES:
        maf_label = f"MAF{maf}"
        
        for vtype in VAR_TYPES:
            suffix = "" if vtype == "" else "_coding"
            pfx = f"meta_{anc}_{maf_label}{suffix}.autosomes"
            
            print(f"\n{'='*70}")
            print(f"Processing: {anc} - {maf_label} - {vtype}")
            print(f"{'='*70}")
            
            OUTDIR = f"{OUTDIR_BASE}/{anc}/{maf_label}/{vtype}"
            
            bash_commands = f"""
set -euxo pipefail

export PATH=$HOME/bin:$PATH

echo "Running REGENIE Step 2 for {anc}, {maf_label}, type {vtype}"

# Check input files exist
echo "Checking input files..."
ls -lh {pfx}.pgen {pfx}.pvar {pfx}.psam
ls -lh {pheno_file}
ls -lh {covar_file}
ls -lh {pred_file}
ls -lh {pfx}_anno_proper4.txt
ls -lh {pfx}_sets_proper4.txt
ls -lh {pfx}_mask_proper4.txt

# Run REGENIE Step 2
~/bin/regenie \\
  --step 2 \\
  --pgen {pfx} \\
  --phenoFile {pheno_file} \\
  --covarFile {covar_file} \\
  --phenoCol PHENO \\
  --covarColList SEX,Age_at_Onset,PCA1,PCA2,PCA3,PCA4,PCA5,PCA6,PCA7,PCA8,PCA9,PCA10 \\
  --bt \\
  --cc12 \\
  --pred {pred_file} \\
  --anno-file {pfx}_anno_proper4.txt \\
  --set-list {pfx}_sets_proper4.txt \\
  --mask-def {pfx}_mask_proper4.txt \\
  --vc-tests skato \\
  --threads {THREADS} \\
  --gz \\
  --out results_step2_{anc}_{maf_label}_{vtype}

echo "REGENIE Step 2 finished"

echo "Uploading results to {OUTDIR}"

gsutil -m cp results_step2_{anc}_{maf_label}_{vtype}* {OUTDIR}/ || true

echo "done: Upload complete for {anc} {maf_label} {vtype}"
"""
            
            # Run the bash commands
            try:
                subprocess.run(bash_commands, shell=True, check=True, executable="/bin/bash")
                print(f"done: Successfully completed: {anc} - {maf_label} - {vtype}")
            except subprocess.CalledProcessError as e:
                print(f"ERROR: Failed: {anc} - {maf_label} - {vtype}")
                print(f"Error: {e}")
                # Continue with next combination instead of stopping
                continue
